# 06 — SHAP vs LIME

- **Manual SHAP** is the primary explanation method for the tree-based XGBoost model.
- **LIME** is kept as a comparison baseline and as an educational local-surrogate method.
- The notebook exports:
  - one global SHAP summary plot,
  - several local SHAP figures,
  - several local LIME figures,
  - one side-by-side SHAP vs LIME comparison on the same anomalous trip.


# 0. Config and shared helpers


In [ ]:
from __future__ import annotations

import json
import math
import sys
import warnings
import zipfile
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.options.display.float_format = lambda x: f"{x:.4f}"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()

    candidates = [start] + list(start.parents)
    for cand in candidates:
        if (cand / "regression_model").exists() and (cand / "notebooks").exists():
            return cand
        if (cand / "README.md").exists() and (cand / "src").exists():
            return cand
    return start

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

# Prefer the baseline artifact because the artifact shipped inside output.zip
# can be version-sensitive across pandas / xgboost installations.
BASELINE_ARTIFACT_PATH = PROJECT_ROOT / "regression_model" / "models" / "xgb_energy_artifact.joblib"
OUTPUT_ZIP_PATH = PROJECT_ROOT / "notebooks/baseline_implementation" / "output.zip"

BLOG_OUTPUT_DIR = PROJECT_ROOT / "outputs_blogpost"
FIG_DIR = BLOG_OUTPUT_DIR / "figures"
TABLE_DIR = BLOG_OUTPUT_DIR / "tables"

for p in [BLOG_OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_NUMERIC_FEATURES = [
    "duration_min",
    "distance_km",
    "speed_mean",
    "speed_var",
    "accel_mean",
    "accel_var",
    "accel_p95",
    "stop_go_ratio",
    "idle_time_min",
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
    "hv_current_abs_mean",
    "hv_current_abs_p95",
    "hv_voltage_mean",
    "maf_mean",
    "maf_p95",
    "Generalized_Weight",
]

DIRECT_TARGET_COMPONENTS = [
    "fuel_energy_kWh",
    "battery_energy_kWh",
    "ac_energy_kWh",
    "heater_energy_kWh",
]

CATEGORICAL_FEATURES = [
    "VehicleType",
    "Vehicle Class",
    "Transmission",
    "Drive Wheels",
]

TARGET = "energy_per_km"
YHAT_COL = "predicted_energy_per_km"
RESIDUAL_COL = "residual"

def load_scored_table_from_repo() -> pd.DataFrame:
    if not OUTPUT_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {OUTPUT_ZIP_PATH}. Place these notebooks into the repository root."
        )

    with zipfile.ZipFile(OUTPUT_ZIP_PATH) as zf:
        inner = "outputs_final/cache/trip_table_scored.csv.gz"
        if inner not in zf.namelist():
            raise FileNotFoundError(f"{inner} not found inside {OUTPUT_ZIP_PATH}")
        raw_gz = zf.read(inner)

    import gzip
    csv_bytes = gzip.decompress(raw_gz)
    from io import BytesIO
    df = pd.read_csv(BytesIO(csv_bytes))
    return df

def load_baseline_artifact() -> dict:
    if not BASELINE_ARTIFACT_PATH.exists():
        raise FileNotFoundError(f"Could not find baseline artifact at {BASELINE_ARTIFACT_PATH}")
    return joblib.load(BASELINE_ARTIFACT_PATH)

def safe_numeric_fill_value(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce")
    median = s.median()
    if not np.isfinite(median):
        return 0.0
    return float(median)

def fill_raw_features(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        numeric_fill: dict[str, float] | None = None,
    ) -> pd.DataFrame:

    selected_columns = list(model_numeric_features) + list(categorical_features)
    out = df.reindex(columns=selected_columns).copy()

    for col in model_numeric_features:
        if col not in out.columns:
            out[col] = np.nan
    for col in categorical_features:
        if col not in out.columns:
            out[col] = "NO DATA"

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(out[col]) for col in model_numeric_features}

    for col in model_numeric_features:
        fill_value = numeric_fill.get(col, 0.0)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(fill_value)

    for col in categorical_features:
        out[col] = out[col].astype("string").fillna("NO DATA")

    return out

def make_design(
        df: pd.DataFrame,
        model_numeric_features: list[str],
        categorical_features: list[str],
        design_columns: list[str] | None = None,
        numeric_fill: dict[str, float] | None = None,
    ) -> tuple[pd.DataFrame, dict[str, float], list[str]]:

    raw = fill_raw_features(
        df,
        model_numeric_features=model_numeric_features,
        categorical_features=categorical_features,
        numeric_fill=numeric_fill,
    )
    
    X = pd.get_dummies(raw, columns=categorical_features, dummy_na=False)

    if design_columns is None:
        design_columns = list(X.columns)
    X = X.reindex(columns=design_columns, fill_value=0.0)

    if numeric_fill is None:
        numeric_fill = {col: safe_numeric_fill_value(raw[col]) for col in model_numeric_features}

    return X.astype(float), numeric_fill, list(design_columns)

def evaluate_regression(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

def transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if transform is None:
        return y
    if transform == "log1p":
        if np.any(y < 0):
            raise ValueError("log1p requires non-negative targets")
        return np.log1p(y)
    raise ValueError(f"Unsupported transform: {transform}")

def inverse_transform_target(y: np.ndarray, transform: str | None) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if transform is None:
        return y
    if transform == "log1p":
        out = np.expm1(y)
        return np.clip(out, 0.0, None)
    raise ValueError(f"Unsupported transform: {transform}")


from src.xai.lime import explain_instance

# Notebook-specific output folders
XAI_DIR = BLOG_OUTPUT_DIR / "xai"
SHAP_DIR = XAI_DIR / "shap"
LIME_DIR = XAI_DIR / "lime"
COMPARE_DIR = XAI_DIR / "comparisons"

for p in [XAI_DIR, SHAP_DIR, LIME_DIR, COMPARE_DIR]:
    p.mkdir(parents=True, exist_ok=True)


## 1. Load the scored table and rebuild the aligned model design

We use the same cleaned feature definition as in the final baseline notebook:
direct target-construction energy components are excluded from the model design.


In [ ]:
scored_df = load_scored_table_from_repo()
artifact = load_baseline_artifact()

MODEL_NUMERIC_FEATURES = [c for c in RAW_NUMERIC_FEATURES if c not in DIRECT_TARGET_COMPONENTS]

X_all, _, _ = make_design(
    scored_df,
    model_numeric_features=MODEL_NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    design_columns=artifact["feature_columns"],
    numeric_fill=artifact["feature_medians"],
)

feature_columns = artifact["feature_columns"]
model = artifact["model"]
background_design = artifact["lime_background"].copy()

test_mask = scored_df["split"].eq("test")
test_scored_df = scored_df.loc[test_mask].reset_index(drop=True)
X_test = X_all.loc[test_mask].reset_index(drop=True)

print("X_test shape:", X_test.shape)
display(test_scored_df.head(2))


## 2. Select anomalous trips for explanation

For a clean final story, we explain the strongest anomalous trips in the **test split**.
We also drop duplicate vehicles so that the cases are diverse.


In [ ]:
N_CASES = 5

explain_cases = (
    test_scored_df.loc[test_scored_df["is_anomaly"]]
    .sort_values(RESIDUAL_COL, ascending=False)
    .drop_duplicates(subset=["VehId"])
    .head(N_CASES)
    .reset_index(drop=True)
)

display(
    explain_cases.loc[:, ["trip_id", "VehId", TARGET, YHAT_COL, RESIDUAL_COL, "VehicleType", "Transmission"]]
)


# 3. Local SHAP implementation

In [ ]:
import math
import itertools
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


class ManualSHAP:
    """
    Manual SHAP-style explainer.

    Important:
    - It supports:
        * exact Shapley values on a small feature subset
        * permutation-based approximate SHAP for practical use
        * global importance and local plots
    """

    def __init__(
            self,
            model,
            background: pd.DataFrame,
            feature_names: Optional[list[str]] = None,
            random_state: int = 42,
            max_background_rows: Optional[int] = 64,
        ):

        if not isinstance(background, pd.DataFrame):
            raise TypeError("background must be a pandas DataFrame")

        if background.empty:
            raise ValueError("background must not be empty")

        self.model = model
        self.random_state = random_state

        if max_background_rows is not None and len(background) > max_background_rows:
            self.background = background.sample(
                max_background_rows,
                random_state=random_state
            ).reset_index(drop=True)
        else:
            self.background = background.reset_index(drop=True).copy()

        self.feature_names = feature_names or list(self.background.columns)

        missing = [c for c in self.feature_names if c not in self.background.columns]
        if missing:
            raise ValueError(f"Missing features in background: {missing}")

        self.background = self.background[self.feature_names].copy()
        self.feature_to_idx = {f: i for i, f in enumerate(self.feature_names)}

        if hasattr(model, "feature_importances_"):
            self.feature_importance_series = pd.Series(
                model.feature_importances_,
                index=self.feature_names
            ).sort_values(ascending=False)

        else:
            self.feature_importance_series = pd.Series(
                np.ones(len(self.feature_names), dtype=float),
                index=self.feature_names
            ).sort_values(ascending=False)

        # Cache base value once
        self._base_value_cached = float(
            np.asarray(self.model.predict(self.background)).reshape(-1).mean()
        )

    # ---------------------------------------------------------
    # Internal helpers
    # ---------------------------------------------------------
    def _ensure_series(self, x0: pd.Series | pd.DataFrame) -> pd.Series:
        if isinstance(x0, pd.DataFrame):
            if len(x0) != 1:
                raise ValueError("If x0 is a DataFrame, it must contain exactly one row.")
            x0 = x0.iloc[0]

        if not isinstance(x0, pd.Series):
            raise TypeError("x0 must be a pandas Series or a single-row DataFrame")

        missing = [c for c in self.feature_names if c not in x0.index]
        if missing:
            raise ValueError(f"x0 is missing required features: {missing}")

        return x0[self.feature_names].copy()

    def _predict_df(self, X: pd.DataFrame) -> np.ndarray:
        preds = self.model.predict(X)
        return np.asarray(preds).reshape(-1)

    def _coalition_value(
            self,
            x0: pd.Series,
            coalition: set[str],
        ) -> float:

        """
        Interventional coalition value:
        - fix coalition features to x0
        - fill all other features from background rows
        - average the model prediction over background
        """
        X_masked = self.background.copy()

        for col in coalition:
            X_masked[col] = x0[col]

        preds = self._predict_df(X_masked)
        return float(preds.mean())

    def _resolve_feature_subset(
        self,
        features: Optional[list[str]] = None,
        top_features: Optional[int] = None,
    ) -> list[str]:
        if features is not None:
            for f in features:
                if f not in self.feature_names:
                    raise ValueError(f"Unknown feature: {f}")
            return list(features)

        if top_features is not None:
            return list(self.feature_importance_series.head(top_features).index)

        return list(self.feature_names)

    # ---------------------------------------------------------
    # Exact SHAP from definition
    # ---------------------------------------------------------
    def exact_shap_row(
        self,
        x0: pd.Series | pd.DataFrame,
        features: Optional[list[str]] = None,
        top_features: Optional[int] = 6,
    ) -> pd.Series:
        """
        Exact Shapley values from the combinatorial definition.

        Use only for a small feature subset.

        Optimized with memoization of coalition values for the current row.
        """
        x0 = self._ensure_series(x0)
        features = self._resolve_feature_subset(features=features, top_features=top_features)

        M = len(features)
        factorial_M = math.factorial(M)
        phi = {f: 0.0 for f in features}

        coalition_cache: dict[frozenset[str], float] = {
            frozenset(): self._base_value_cached
        }

        def v(S: set[str]) -> float:
            key = frozenset(S)
            if key not in coalition_cache:
                coalition_cache[key] = self._coalition_value(x0, set(key))
            return coalition_cache[key]

        for j in features:
            others = [f for f in features if f != j]

            for r in range(len(others) + 1):
                for subset in itertools.combinations(others, r):
                    S = set(subset)
                    weight = (
                        math.factorial(len(S))
                        * math.factorial(M - len(S) - 1)
                        / factorial_M
                    )

                    v_S = v(S)
                    v_Sj = v(S | {j})

                    phi[j] += weight * (v_Sj - v_S)

        return pd.Series(phi).sort_values(key=np.abs, ascending=False)

    # ---------------------------------------------------------
    # Permutation SHAP approximation
    # ---------------------------------------------------------
    def permutation_shap_row(
            self,
            x0: pd.Series | pd.DataFrame,
            features: Optional[list[str]] = None,
            top_features: Optional[int] = 15,
            n_permutations: int = 200,
            random_state: Optional[int] = None,
        ) -> pd.Series:

        """
        Approximate SHAP via random feature permutations.

        For each permutation:
        - start from empty coalition
        - add features one by one
        - record each marginal contribution

        Optimized version:
        - background is copied once per permutation
        - coalition is updated incrementally
        """
        x0 = self._ensure_series(x0)
        features = self._resolve_feature_subset(features=features, top_features=top_features)

        rng = np.random.default_rng(
            self.random_state if random_state is None else random_state
        )

        phi = {f: 0.0 for f in features}

        for _ in range(n_permutations):
            perm = list(rng.permutation(features))

            # Start from empty coalition = background baseline
            X_current = self.background.copy()
            prev_value = self._base_value_cached

            for f in perm:
                X_current[f] = x0[f]
                new_value = float(self._predict_df(X_current).mean())
                phi[f] += (new_value - prev_value)
                prev_value = new_value

        for f in phi:
            phi[f] /= n_permutations

        return pd.Series(phi).sort_values(key=np.abs, ascending=False)

    # ---------------------------------------------------------
    # Explain many rows
    # ---------------------------------------------------------
    def explain_many(
            self,
            X: pd.DataFrame,
            method: str = "permutation",
            features: Optional[list[str]] = None,
            top_features: Optional[int] = 15,
            n_permutations: int = 100,
            exact_top_features: int = 6,
        ) -> pd.DataFrame:

        """
        Explain multiple rows and return a DataFrame of SHAP values.

        method:
        - 'permutation'
        - 'exact'
        """
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X must be a pandas DataFrame")

        results = []

        if method == "permutation":
            resolved_features = self._resolve_feature_subset(
                features=features,
                top_features=top_features
            )

        elif method == "exact":
            resolved_features = self._resolve_feature_subset(
                features=features,
                top_features=exact_top_features
            )

        else:
            raise ValueError("method must be 'permutation' or 'exact'")

        for i in range(len(X)):
            row = X.iloc[i]

            if method == "permutation":
                phi = self.permutation_shap_row(
                    x0=row,
                    features=resolved_features,
                    top_features=None,
                    n_permutations=n_permutations,
                    random_state=self.random_state + i,
                )

            else:
                phi = self.exact_shap_row(
                    x0=row,
                    features=resolved_features,
                    top_features=None,
                )

            results.append(phi)

        shap_df = pd.DataFrame(results).fillna(0.0)
        shap_df.index = X.index
        return shap_df

    # ---------------------------------------------------------
    # Base value and additivity helpers
    # ---------------------------------------------------------
    def base_value(self) -> float:
        """
        Expected prediction under the background distribution.
        Equivalent to v(empty set).
        """
        return self._base_value_cached

    def reconstruct_prediction(
            self,
            shap_values_row: pd.Series,
        ) -> float:

        return self._base_value_cached + float(shap_values_row.sum())

    # ---------------------------------------------------------
    # Global importance
    # ---------------------------------------------------------
    def summary_importance(self, shap_df: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame({
            "feature": shap_df.columns,
            "mean_abs_shap": np.abs(shap_df).mean(axis=0).values,
            "mean_shap": shap_df.mean(axis=0).values,
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
        return out

    # ---------------------------------------------------------
    # Plot summary (SHAP-like beeswarm substitute)
    # ---------------------------------------------------------
    def plot_summary(
            self,
            shap_df: pd.DataFrame,
            X_values: pd.DataFrame,
            top_n: int = 20,
            title: str = "Manual SHAP summary",
            save_path: Optional[str] = None,
            figsize: tuple[int, int] = (10, 7),
        ) -> None:

        """
        SHAP-like summary scatter plot.

        shap_df: rows = objects, cols = features
        X_values: same rows, same feature columns
        """
        if not isinstance(shap_df, pd.DataFrame):
            raise TypeError("shap_df must be a pandas DataFrame")
        
        if not isinstance(X_values, pd.DataFrame):
            raise TypeError("X_values must be a pandas DataFrame")

        common_cols = [c for c in shap_df.columns if c in X_values.columns]

        if len(common_cols) == 0:
            raise ValueError("No overlapping columns between shap_df and X_values")

        shap_df = shap_df[common_cols].copy()
        X_values = X_values[common_cols].copy()

        imp = np.abs(shap_df).mean(axis=0).sort_values(ascending=False)
        selected = list(imp.head(top_n).index)

        shap_plot_df = shap_df[selected]
        X_plot_df = X_values[selected]

        order = list(imp.head(top_n).index)
        order_reversed = order[::-1]

        plt.figure(figsize=figsize)
        rng = np.random.default_rng(self.random_state)

        for yi, feat in enumerate(order_reversed):
            shap_col = shap_plot_df[feat].to_numpy()
            feat_vals = X_plot_df[feat].to_numpy()

            feat_min = np.nanmin(feat_vals)
            feat_max = np.nanmax(feat_vals)
            feat_range = feat_max - feat_min

            if feat_range > 1e-12:
                color_vals = (feat_vals - feat_min) / (feat_range + 1e-12)

            else:
                color_vals = np.full(len(feat_vals), 0.5, dtype=float)

            jitter = rng.normal(0, 0.10, size=len(shap_col))

            plt.scatter(
                shap_col,
                np.full(len(shap_col), yi) + jitter,
                c=color_vals,
                cmap="coolwarm",
                alpha=0.7,
                s=18,
                edgecolors="none",
            )

        plt.axvline(0.0, linewidth=1)
        plt.yticks(range(len(order_reversed)), order_reversed)
        plt.xlabel("Contribution to prediction")
        plt.ylabel("Feature")
        plt.title(title)
        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=220, bbox_inches="tight")

        plt.show()

    # ---------------------------------------------------------
    # Plot local explanation
    # ---------------------------------------------------------
    def plot_local(
            self,
            shap_row: pd.Series,
            x0: Optional[pd.Series] = None,
            top_n: int = 10,
            title: str = "Manual SHAP local explanation",
            save_path: Optional[str] = None,
            figsize: tuple[int, int] = (9, 5),
        ) -> None:

        """
        Horizontal bar plot for one explained object.
        """
        shap_row = shap_row.sort_values(key=np.abs, ascending=False).head(top_n)
        plot_series = shap_row.sort_values()

        plt.figure(figsize=figsize)
        plt.barh(plot_series.index, plot_series.values)
        plt.axvline(0.0, linewidth=1)
        plt.xlabel("Contribution to prediction")
        plt.title(title)
        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=220, bbox_inches="tight")

        plt.show()

    # ---------------------------------------------------------
    # Convenience method for one-row report
    # ---------------------------------------------------------
    def explain_row_report(
            self,
            x0: pd.Series | pd.DataFrame,
            method: str = "permutation",
            features: Optional[list[str]] = None,
            top_features: Optional[int] = 15,
            n_permutations: int = 200,
        ) -> pd.DataFrame:

        """
        Return a neat report DataFrame with:
        - feature
        - phi
        - x0 value
        """
        x0_series = self._ensure_series(x0)

        if method == "permutation":
            phi = self.permutation_shap_row(
                x0=x0_series,
                features=features,
                top_features=top_features,
                n_permutations=n_permutations,
            )

        elif method == "exact":
            phi = self.exact_shap_row(
                x0=x0_series,
                features=features,
                top_features=top_features,
            )
            
        else:
            raise ValueError("method must be 'permutation' or 'exact'")

        report = pd.DataFrame({
            "feature": phi.index,
            "phi": phi.values,
            "x0_value": [x0_series[f] for f in phi.index],
        })

        report["abs_phi"] = report["phi"].abs()
        report["direction"] = np.where(report["phi"] >= 0, "increase", "decrease")
        return report.sort_values("abs_phi", ascending=False).reset_index(drop=True)

## 3. Global Manual SHAP summary

A global plot is valuable for the blog because it tells the reader which features
matter *across* many trips before we zoom into local case studies.


In [ ]:
# ============================================================
# Manual SHAP-style global summary
# ============================================================

MAX_SHAP_ROWS = 80
TOP_FEATURES = 12
N_PERMUTATIONS = 40

X_shap = X_test.head(MAX_SHAP_ROWS).copy()

manual_shap = ManualSHAP(
    model=model,
    background=X_test,
    random_state=42,
    max_background_rows=64,
)

# shap_values here is a DataFrame: rows = objects, cols = features
shap_values = manual_shap.explain_many(
    X=X_shap,
    method="permutation",
    top_features=TOP_FEATURES,
    n_permutations=N_PERMUTATIONS,
)

shap_summary_path = SHAP_DIR / "shap_summary.png"

manual_shap.plot_summary(
    shap_df=shap_values,
    X_values=X_shap[shap_values.columns],
    top_n=min(TOP_FEATURES, len(shap_values.columns)),
    title="Manual permutation SHAP summary",
    save_path=str(shap_summary_path),
)

print("Saved:", shap_summary_path)

manual_global_importance = manual_shap.summary_importance(shap_values)

display(manual_global_importance.head(20))

## 4. Helper functions for local SHAP and local LIME

To keep the final exported figures readable, we convert each local explanation into
a simple horizontal bar chart of top absolute contributions.


In [ ]:
def plot_local_contributions(
        contrib_df: pd.DataFrame,
        feature_col: str,
        value_col: str,
        title: str,
        save_path: Path,
        top_k: int = 10,
    ):

    plot_df = contrib_df.copy()
    plot_df["abs_value"] = plot_df[value_col].abs()
    plot_df = plot_df.sort_values("abs_value", ascending=False).head(top_k).iloc[::-1]

    plt.figure(figsize=(8, max(4, 0.40 * len(plot_df))))
    plt.barh(plot_df[feature_col], plot_df[value_col])
    plt.xlabel("Contribution")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def local_shap_df(
        row_index: int,
        X_local: pd.DataFrame,
        shap_values: np.ndarray,
    ) -> pd.DataFrame:
    
    vals = shap_values[row_index]
    return pd.DataFrame({
        "feature": X_local.columns,
        "contribution": vals,
    })

def black_box_predict_design(df_design: pd.DataFrame) -> np.ndarray:
    aligned = df_design.reindex(columns=feature_columns, fill_value=0.0)
    return np.asarray(model.predict(aligned), dtype=float)


## 5. Export local SHAP and local LIME explanations for the selected trips


In [ ]:
def local_shap_df_from_manual(row_idx, shap_values_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert one row of manual SHAP values into the same tidy format:
    columns = ['feature', 'contribution']
    """
    local_series = shap_values_df.loc[row_idx].sort_values(key=np.abs, ascending=False)

    return pd.DataFrame({
        "feature": local_series.index,
        "contribution": local_series.values,
    }).reset_index(drop=True)

In [ ]:
lime_rows = []
shap_rows = []
comparison_trip_ids = []

# ============================================================
# Manual SHAP precomputation for the full test set
# ============================================================

# Manual SHAP values for all test rows we may want to use
full_test_shap_values = manual_shap.explain_many(
    X=X_test,
    method="permutation",
    top_features=TOP_FEATURES,
    n_permutations=N_PERMUTATIONS,
)

for _, case_row in explain_cases.iterrows():
    trip_id = str(case_row["trip_id"])
    idx = int(test_scored_df.index[test_scored_df["trip_id"].astype(str) == trip_id][0])

    # ---------- Local manual SHAP ----------
    shap_df = local_shap_df_from_manual(idx, full_test_shap_values)
    shap_path = SHAP_DIR / f"trip_{trip_id}_shap.png"
    shap_csv_path = SHAP_DIR / f"trip_{trip_id}_shap.csv"

    plot_local_contributions(
        shap_df,
        feature_col="feature",
        value_col="contribution",
        title=f"Manual SHAP explanation for trip {trip_id}",
        save_path=shap_path,
        top_k=10,
    )

    shap_df.to_csv(shap_csv_path, index=False)

    shap_rows.append({
        "trip_id": trip_id,
        "method": "ManualSHAP",
        "top_features": " | ".join(
            shap_df.assign(abs_value=lambda d: d["contribution"].abs())
            .sort_values("abs_value", ascending=False)
            .head(10)
            .apply(lambda r: f"{r['feature']}:{r['contribution']:.4f}", axis=1)
            .tolist()
        ),
    })

    # ---------- Local LIME ----------
    x0_design = X_test.loc[idx].copy()

    lime_exp = explain_instance(
        trip_id=trip_id,
        x0=x0_design,
        background_df=background_design,
        feature_cols=feature_columns,
        black_box_predict=black_box_predict_design,
        n_samples=4000,
        kernel_width=0.75 * np.sqrt(max(len(feature_columns), 1)),
        ridge_alpha=0.1,
        top_k=10,
        random_state=RANDOM_STATE,
    )

    lime_df = pd.DataFrame(lime_exp.top_features, columns=["feature", "contribution"])
    lime_path = LIME_DIR / f"trip_{trip_id}_lime.png"
    lime_csv_path = LIME_DIR / f"trip_{trip_id}_lime.csv"

    plot_local_contributions(
        lime_df,
        feature_col="feature",
        value_col="contribution",
        title=f"LIME explanation for trip {trip_id}",
        save_path=lime_path,
        top_k=10,
    )
    
    lime_df.to_csv(lime_csv_path, index=False)

    lime_rows.append({
        "trip_id": trip_id,
        "method": "LIME",
        "local_r2": lime_exp.local_r2,
        "local_rmse": lime_exp.local_rmse,
        "top_features": " | ".join(
            lime_df.head(10).apply(
                lambda r: f"{r['feature']}:{r['contribution']:.4f}",
                axis=1
            ).tolist()
        ),
    })

    comparison_trip_ids.append(trip_id)

print("Saved local manual SHAP explanations to:", SHAP_DIR)
print("Saved local LIME explanations to:", LIME_DIR)

## 6. SHAP vs LIME on the same trip

We now create one explicit side-by-side comparison for the strongest anomalous case.
This is one of the key figures for the final blog post and the presentation.


In [ ]:
trip_id = comparison_trip_ids[0]

shap_df = pd.read_csv(SHAP_DIR / f"trip_{trip_id}_shap.csv")
lime_df = pd.read_csv(LIME_DIR / f"trip_{trip_id}_lime.csv")

shap_plot = (
    shap_df.assign(abs_value=lambda d: d["contribution"].abs())
    .sort_values("abs_value", ascending=False)
    .head(10)
    .loc[:, ["feature", "contribution"]]
    .rename(columns={"contribution": "Manual SHAP"})
)

lime_plot = (
    lime_df.assign(abs_value=lambda d: d["contribution"].abs())
    .sort_values("abs_value", ascending=False)
    .head(10)
    .loc[:, ["feature", "contribution"]]
    .rename(columns={"contribution": "LIME"})
)

compare_df = pd.merge(shap_plot, lime_plot, on="feature", how="outer").fillna(0.0)
compare_df["abs_total"] = compare_df["Manual SHAP"].abs() + compare_df["LIME"].abs()
compare_df = compare_df.sort_values("abs_total", ascending=False).head(12).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, max(5, 0.42 * len(compare_df))))
y = np.arange(len(compare_df))
bar_h = 0.38

ax.barh(y - bar_h / 2, compare_df["Manual SHAP"], height=bar_h, label="Manual SHAP")
ax.barh(y + bar_h / 2, compare_df["LIME"], height=bar_h, label="LIME")
ax.set_yticks(y)
ax.set_yticklabels(compare_df["feature"])
ax.set_xlabel("Contribution")
ax.set_title(f"Manual SHAP vs LIME for trip {trip_id}")
ax.legend()
plt.tight_layout()

compare_path = COMPARE_DIR / f"shap_vs_lime_{trip_id}.png"
plt.savefig(compare_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", compare_path)


## 7. Export explanation summary tables

In [ ]:
shap_summary_df = pd.DataFrame(shap_rows)
lime_summary_df = pd.DataFrame(lime_rows)

case_summary = explain_cases.loc[:, ["trip_id", "VehId", TARGET, YHAT_COL, RESIDUAL_COL, "VehicleType", "Transmission"]].copy()
case_summary = case_summary.merge(lime_summary_df.loc[:, ["trip_id", "local_r2", "local_rmse"]], on="trip_id", how="left")

display(case_summary)
display(lime_summary_df)

shap_summary_df.to_csv(TABLE_DIR / "shap_case_summary.csv", index=False)
lime_summary_df.to_csv(TABLE_DIR / "lime_case_summary.csv", index=False)
case_summary.to_csv(TABLE_DIR / "xai_case_summary.csv", index=False)
